In [22]:
import pandas as pd
import os
import re

In [23]:
def process_sncf_local(directory_path):
    routes_path = os.path.join(directory_path, 'routes.txt')
    trips_path = os.path.join(directory_path, 'trips.txt')

    if not os.path.exists(routes_path) or not os.path.exists(trips_path):
        print("Erreur : fichiers introuvables")
        return

    try:
        df_routes = pd.read_csv(routes_path)
        df_trips = pd.read_csv(trips_path)

        
        def extract_cities(name):
            if pd.isna(name):
                return "", ""
            parts = str(name).split(' - ')
            if len(parts) >= 2:
                origin = parts[0].strip()
                destination = parts[-1].strip()
            else:
                origin = name.strip()
                destination = ""
            
            # Supprime les préfixes comme "22. ", "41+. ", "3b. "
            prefix_pattern = r'^[\d\w]+\+?\.\s*'
            origin = re.sub(prefix_pattern, '', origin)
            destination = re.sub(prefix_pattern, '', destination)
            
            return origin, destination

        df_routes[['origin_city', 'destination_city']] = df_routes['route_long_name'].apply(
            lambda x: pd.Series(extract_cities(x))
        )

        def extract_date(headsign):
            match = re.search(r'(\d{8})$', str(headsign))
            return match.group(1) if match else None

        df_trips['trip_date_str'] = df_trips['trip_id'].apply(extract_date)
        
        df_trips = df_trips.dropna(subset=['trip_date_str'])


        df_trips['date_dt'] = pd.to_datetime(df_trips['trip_date_str'], format='%Y%m%d')
        df_trips['year_week'] = df_trips['date_dt'].dt.strftime('%G-W%V')

        df_weekly = df_trips.groupby(['route_id', 'year_week']).size().reset_index(name='weekly_train')


        df_final = pd.merge(
            df_weekly, 
            df_routes[['route_id', 'origin_city', 'destination_city']], 
            on='route_id', 
            how='left'
        )

        df_final['data_source'] = 'France'

        def get_desserte_type(count):
            if count < 7:
                return "Sous-desservi"
            elif 7 <= count <= 56:
                return "Desserte Normale"
            else:
                return "Bien desservi"

        df_final['desserte_type'] = df_final['weekly_train'].apply(get_desserte_type)

        final_cols = [
            'data_source', 'route_id', 'origin_city', 
            'destination_city', 'weekly_train', 'desserte_type'
        ]
        df_final = df_final[
            (df_final['origin_city'] != '-') & 
            (df_final['origin_city'] != '') & 
            (df_final['destination_city'] != '')
        ]
        
        # prendre ça pour la db
        df_final = df_final[final_cols]

        # output_file = 'sncf_routes_stats.csv'
        # df_final.to_csv(output_file, index=False, encoding='utf-8')

        # print(f"Routes traitées : {df_final['route_id'].nunique()}")
        # print(f"Lignes totales créées : {len(df_final)}")
        # print("\nRépartition des types de desserte :")
        # print(df_final['desserte_type'].value_counts().to_string())

    except Exception as e:
        print(f"Une erreur est survenue : {e}")


In [24]:
dossier_gtfs = '../../../data/france/export-opendata-sncf-gtfs'

if __name__ == "__main__":
    process_sncf_local(dossier_gtfs)

--- Début du traitement des fichiers dans : ../../../data/france/export-opendata-sncf-gtfs ---
Fichiers chargés avec succès.
Routes traitées : 611
Lignes totales créées : 4465

Répartition des types de desserte :
desserte_type
Sous-desservi       2781
Desserte Normale    1638
Bien desservi         46

Fichier généré : sncf_routes_stats.csv
